## Before you run this notebook

- **What you need:** an OpenRouter API key (for the chat and embedding models) and a LangSmith API key (to pull the RAG prompt from the LangSmith Hub). Put them in a `.env` file as `OPENROUTER_API_KEY=...` and `LANGSMITH_API_KEY=...`. If the OpenRouter key is missing, the setup cell asks you to type it in. The LangSmith key is needed by the prompt pull in cell 15. Never commit your key to Git. The `.env` file is listed in `.gitignore`, so Git ignores it by default, but check `git status` before you push.
- **Model:** The `RAG_MODEL` variable in `.env` chooses the chat model. If it is not set, the notebook uses `openai/gpt-5-nano`. Model names are listed at [openrouter.ai/models](https://openrouter.ai/models). They can change, so check there if the default stops working.
- **Privacy:** Your questions and the retrieved text are sent through OpenRouter to the model provider, and the embedding requests are sent there too. Do not include personal or confidential information.


In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv(override=True)

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")

# Model used for every chat call in this notebook. Set RAG_MODEL to override,
# or swap the name on openrouter.ai/models.
RAG_MODEL = os.getenv("RAG_MODEL", "openai/gpt-5-nano")

# Retrieval Augmented Generation (RAG)

RAG gives a large language model (LLM) access to information it was not trained on. Before the model answers, the system **retrieves** relevant text from a set of documents, **augments** the question with that text, and then the LLM **generates** an answer that uses it.

RAG was introduced by researchers at Facebook AI, now part of Meta AI, in 2020. Read their [blog post](https://ai.meta.com/blog/retrieval-augmented-generation-streamlining-the-creation-of-intelligent-natural-language-processing-models/).

## 1. Why RAG exists

LLMs write fluent text, but they have three weaknesses:

- **Static knowledge:** they only know what was in their training data, so they can be out of date.
- **Hallucination:** they can give a confident answer that is wrong.
- **Cost:** retraining a model to update what it knows is expensive.

RAG reduces these problems by giving the model relevant text at question time. It does not remove them. The answer is only as good as the sources and the retrieval step.

## 2. How RAG works

1. The user asks a question.
2. The system searches a set of documents for passages that are similar in meaning to the question.
3. Those passages are added to the prompt as context.
4. The LLM writes an answer based on the question and the context.

<img src="https://docs.aws.amazon.com/images/sagemaker/latest/dg/images/jumpstart/jumpstart-fm-rag.jpg">

## 3. The components in this notebook

| Component | What it does | Where it is in this notebook |
| --- | --- | --- |
| Document loader | Reads text from a source, here web pages | Cell 4 (`WebBaseLoader`) |
| Document chunker | Splits long text into smaller pieces | Cell 7 (`RecursiveCharacterTextSplitter`) |
| Embedder | Turns text into a vector, a list of numbers that represents its meaning | Cell 9 (`OpenAIEmbeddings`) |
| Vector store | Stores the vectors with their text, for fast search | Cell 9 (`FAISS`) |
| Retriever | Finds the chunks closest in meaning to a question | Cell 12 (`as_retriever()`) |
| Prompt | A template that combines the question and the retrieved context | Cells 15 to 17 (`prompt`) |
| LLM | Writes the final answer | Cell 13 (`ChatOpenRouter`) |

## 4. Three subsystems

- **Index:** load, chunk, embed and store your documents. You do this once (cells 4 to 9).
- **Retrieval:** embed the question and find the closest chunks (cells 10 to 12).
- **Augment:** put the question and the chunks into a prompt, then ask the LLM to answer (cells 13 to 19).

<img src="https://python.langchain.com/assets/images/rag_indexing-8160f90a90a33253d0154659cf7d453f.png">

[Build a RAG agent with LangChain](https://docs.langchain.com/oss/python/langchain/rag)

<img src="https://mintcdn.com/langchain-5e9cc07a/I6RpA28iE233vhYX/images/rag_indexing.png?w=1100&fit=max&auto=format&n=I6RpA28iE233vhYX&q=85&s=675f55e100bab5e2904d27db01775ccc">

<img src="https://mintcdn.com/langchain-5e9cc07a/I6RpA28iE233vhYX/images/rag_retrieval_generation.png?w=1100&fit=max&auto=format&n=I6RpA28iE233vhYX&q=85&s=d390a6a758e688ec36352d30b22249b0">


# Load documents

LangChain has many document loaders. Each one reads a source (a file, a folder, a web page, a video transcript) and returns a list of `Document` objects. Check [the documentation](https://docs.langchain.com/oss/python/integrations/document_loaders/index#common-file-types) to see what is available.

## Understanding document loaders

- A **Document** is a piece of text (`page_content`) with extra information about it (`metadata`), such as the source URL.
- Loaders fetch data and return it as Document objects. The main method is `load()`.
- Some loaders support **lazy loading**, which reads data only when you need it instead of all at once.

## How this notebook uses them

This notebook loads web pages with `WebBaseLoader` from `langchain_community.document_loaders`. Instead of a folder path, you give it a URL:

```python
from langchain_community.document_loaders import WebBaseLoader

docs = WebBaseLoader("https://www.coursera.org/articles/ai-trends").load()
```

Always check what you loaded. Not every page returns usable text. Some sites build their content with JavaScript, so a loader that only downloads the HTML can get a message such as "Please enable JS" instead of the article. An earlier version of this notebook loaded a Forbes article that did exactly this, so the source was replaced. Check `len(doc.page_content)` or print the first document before you trust a source. Also check a site's terms of use before you scrape it, especially at scale.


In [2]:
from langchain_community.document_loaders import WebBaseLoader

coursera_2026_trends = WebBaseLoader("https://www.coursera.org/articles/ai-trends").load()

google_trends = WebBaseLoader("https://trends.withgoogle.com/trends/us/artificial-intelligence-search-trends/?hl=en-US").load()

forbes_gen_ai_trends = WebBaseLoader("https://en.wikipedia.org/wiki/Generative_artificial_intelligence").load()

/var/folders/vl/m6g01gp54cl4409b678xyrkm0000gn/T/ipykernel_23998/376273327.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
/Users/moche/IADT_Local/26_27/CC-AIEng/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
/Users/moche/IADT_Local/26_27/CC-AIEng/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, 

In [3]:
forbes_gen_ai_trends[0]

Document(metadata={'source': 'https://www.forbes.com/sites/bernardmarr/2025/10/13/10-generative-ai-trends-in-2026-that-will-transform-work-and-life/', 'title': 'forbes.com', 'language': 'en'}, page_content='forbes.comPlease enable JS and disable any ad blocker')

# Chunk documents

Long documents are hard to search well, and they can be too long for the model to use efficiently. A **text splitter** breaks each document into smaller pieces called **chunks**.

## How the splitter works

1. Split the text at natural boundaries first: paragraphs, then lines, then words.
2. Join the small pieces into chunks until a size limit is reached.
3. Start a new chunk, repeating some of the previous chunk's end (the **overlap**) so that ideas are not cut in half.

## The settings in cell 7

- `RecursiveCharacterTextSplitter`: the default choice. If a piece is still too big, it tries the next separator. Its default separators are `["\n\n", "\n", " ", ""]`.
- `chunk_size=500`: the maximum length of a chunk.
- `chunk_overlap=50`: how many characters neighbouring chunks share.
- `length_function=len`: length is counted in **characters**. Other setups count **tokens** instead.

## The trade-off

- Small chunks are precise, but they can lose the context needed to understand them.
- Large chunks keep more context, but they can mix several topics, so a search may match them for the wrong reason.

There is no perfect size. Change the numbers, re-run cells 7 to 9, and compare the answers.


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    length_function = len
)

coursera_2026_trends_chunks = text_splitter.transform_documents(coursera_2026_trends)

google_trends_chunks = text_splitter.transform_documents(google_trends)

forbes_gen_ai_trends_chunks = text_splitter.transform_documents(forbes_gen_ai_trends)

# Index system

The index system prepares your documents so they can be searched quickly. You do this once, then reuse the result for every question.

## The four steps

1. **Load documents** (cell 4): read the web pages.
2. **Chunk documents** (cell 7): split them into pieces of up to 500 characters, with 50 characters of overlap.
3. **Embed documents** (cell 9): turn each chunk into a vector, a list of numbers that represents its meaning. Chunks with similar meanings get vectors that are close together. The embeddings come from `openai/text-embedding-3-small` through OpenRouter.
4. **Store embeddings** (cell 9): `FAISS` keeps the vectors and their text together, so the retriever can search them.

## Two details to know

- **Caching:** `CacheBackedEmbeddings` saves each embedding in the `./cache/` folder. If the same text is embedded again with the same model, the saved result is reused. This saves API calls and cost.
- **In memory:** the FAISS index lives in memory. If you restart the kernel, run cell 9 again. The cache means the embedding API is not called again for text it has already seen.


In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings  
from langchain_community.vectorstores import FAISS
from langchain_classic.storage import LocalFileStore

store = LocalFileStore("./cache/")

# create an embedder
# Embeddings go through OpenRouter. langchain_openrouter only provides chat models.
core_embeddings_model = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    check_embedding_ctx_length=False,  # send plain text, not token IDs, which OpenRouter needs
)

embedder = CacheBackedEmbeddings.from_bytes_store(
    core_embeddings_model,
    store,
    namespace = core_embeddings_model.model
)

# store embeddings in vector store
vectorstore = FAISS.from_documents(coursera_2026_trends_chunks, embedder)

vectorstore.add_documents(google_trends_chunks)

vectorstore.add_documents(forbes_gen_ai_trends_chunks)

/Users/moche/IADT_Local/26_27/CC-AIEng/.venv/lib/python3.14/site-packages/langchain_classic/embeddings/cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


['34450580-e314-41e0-b875-f34237534998']

# Retrieval system

The retrieval system finds the chunks that are most relevant to a user's question.

## The steps

1. **Obtain the user query:** the question you type, for example "What areas of AI should students consider?"
2. **Embed the query:** the question is turned into a vector with the same embedding model used for the documents. Both sides must use the same model, or their vectors cannot be compared.
3. **Vector search:** the vector store compares the question's vector with every chunk's vector and finds the closest matches.
4. **Return the relevant documents:** by default, `as_retriever()` returns the top 4 chunks (`k=4`).

In this notebook, steps 2 to 4 happen inside the retriever created in cell 12. You do not call the embedder yourself.


In [6]:
from langchain_openrouter import ChatOpenRouter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langsmith import Client


In [7]:
# instantiate a retriever
retriever = vectorstore.as_retriever()

In [8]:
llm = ChatOpenRouter(model=RAG_MODEL, temperature=0)

# Augment system

The augment system combines the question with the retrieved chunks and sends the result to the LLM.

## The chain in cell 15

```python
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
```

Read it from top to bottom. Each `|` passes the output of one step to the next:

1. **Retrieve and format:** `retriever` finds the chunks, and `format_docs` joins their text with blank lines.
2. **Pass the question through:** `RunnablePassthrough()` keeps the question unchanged.
3. **Fill the prompt:** `prompt` places the context and the question into a template (explained in the next section).
4. **Call the LLM:** `llm` sends the filled prompt to the model. `temperature=0` makes the output less random, but it does not guarantee the same answer every time.
5. **Parse the output:** `StrOutputParser()` turns the model's reply into plain text.

The prompt asks the model to answer in at most three sentences and to say when it does not know. These are instructions the model tries to follow. They are not guarantees, so check the answer against the sources.


## The prompt template

A **prompt template** is text with blanks (placeholders) that are filled in when the code runs. This one has two:

- `{context}`: the text of the retrieved chunks.
- `{question}`: the user's question.

The next cell:

- Defines `format_docs`, which joins the text of the chunks.
- Connects to LangSmith with `Client()`. This needs `LANGSMITH_API_KEY` (see the setup section at the top).
- Pulls the template `moche/rag-prompt` from the LangSmith Hub with `pull_prompt`. Keeping the prompt in the Hub lets you change its wording without editing the notebook.

The code comment says the `dangerously_pull_public_prompt=True` flag is needed by langsmith 0.14 and later when pulling by owner and name.

Cell 16 prints the template. Cell 17 lists the input variables, which must match the placeholders.


In [9]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

client = Client()
# langsmith 0.14+ needs this flag to pull by owner/name. It is our own plain-text prompt template.
prompt = client.pull_prompt("moche/rag-prompt", dangerously_pull_public_prompt=True)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [10]:
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'moche', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': 'cbf9a5588c05a829d7924b7865122580e5b45523ae8fdae9513c1c05de280b04'}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\n\nQuestion: {question} \n\nContext: {context} \n\nAnswer:")

In [11]:
prompt.input_variables

['context', 'question']

In [12]:
# This is the entire augment system!
rag_chain.invoke("What are the trends in AI?")

'In 2026, AI trends include increased adoption of Generative AI (GenAI), AI-enhanced scientific research, and intensified AI regulations.'

In [13]:
rag_chain.invoke("What areas of AI should students consider?")

'Students should consider AI in healthcare and AI in financial services, along with foundational AI topics like machine learning and deep learning. They should also pursue AI fluency/frameworks and hands‑on areas such as AI agents. These areas reflect current trends and growing industry demand.'

## Check the answer

The answers in cells 18 and 19 should be based on the retrieved text, but the model can still add its own knowledge. Before you trust an answer:

1. Find its sources (cells 22 and 24 show them).
2. Check that each claim appears in those sources.
3. Notice when an answer says something the sources do not. This is where hallucination can still happen.

Answers can also change between runs, even with `temperature=0`. Your results may not match the outputs saved in this notebook.


## Return sources

The chain in cell 15 returns only the answer text. To show where the answer came from, we also need the retrieved documents.

Cell 21 does this in two steps:

- `RunnableParallel` runs the retriever and the question at the same time, and returns a dictionary with `context` (the retrieved Documents) and `question`.
- `.assign(answer=rag_chain_from_docs)` adds an `answer` key. `rag_chain_from_docs` works from the documents already retrieved, so the retriever runs once, not twice.

Why not reuse `rag_chain`? It retrieves the chunks internally, so you never see them. Using `rag_chain_from_docs` keeps the chunks and the answer together.


In [14]:
from langchain_core.runnables import RunnableParallel

rag_chain_from_docs = (
    RunnablePassthrough.assign(context=(lambda x: format_docs(x["context"])))
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain_with_source = RunnableParallel({"context": retriever, "question": RunnablePassthrough()}).assign(answer=rag_chain_from_docs)

In [15]:
result = rag_chain_with_source.invoke("What areas of AI should students consider?")

### Display as HTML

Cell 24 formats the result from cell 22 (the question, the answer and the retrieved chunks) as an HTML card, so it is easier to read. This is presentation only. It does not change how RAG works.

The function escapes special characters, such as `<` and `&`, before it puts text into the HTML. This is a basic safety step. Retrieved text comes from web pages that you did not write, so it should be treated as plain text, not as code.


In [16]:
from IPython.display import HTML, display
import html

def format_rag_results_html(result):
    """Format RAG results as beautiful HTML"""
    
    question = result.get('question', '')
    answer = result.get('answer', '')
    context_docs = result.get('context', [])
    
    # Process context documents
    processed_docs = []
    for doc in context_docs:
        if hasattr(doc, 'page_content'):  # Document object (from langchain)
            processed_docs.append({
                'source': doc.metadata.get('source', 'Unknown'),
                'title': doc.metadata.get('title', 'Untitled'),
                'content': doc.page_content
            })
        elif isinstance(doc, dict):
            # Handle dict with metadata key (Document-like structure)
            if 'metadata' in doc:
                processed_docs.append({
                    'source': doc['metadata'].get('source', 'Unknown'),
                    'title': doc['metadata'].get('title', 'Untitled'),
                    'content': doc.get('page_content', '')
                })
            else:
                # Simple dict structure
                processed_docs.append({
                    'source': doc.get('source', 'Unknown'),
                    'title': doc.get('title', 'Untitled'),
                    'content': doc.get('content', doc.get('page_content', ''))
                })
    
    # Escape HTML special characters
    def escape_html(text):
        return html.escape(str(text))
    
    # Generate HTML
    html_content = f"""
    <style>
        .rag-container {{
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Oxygen, Ubuntu, Cantarell, sans-serif;
            max-width: 1200px;
            margin: 20px auto;
            padding: 20px;
            background: #f8f9fa;
            border-radius: 8px;
        }}
        .rag-header {{
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 20px;
            border-radius: 8px;
            margin-bottom: 20px;
        }}
        .rag-header h1 {{
            margin: 0;
            font-size: 24px;
        }}
        .question-box {{
            background: white;
            padding: 20px;
            border-radius: 8px;
            margin-bottom: 20px;
            border-left: 4px solid #667eea;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }}
        .question-box h2 {{
            margin: 0 0 10px 0;
            color: #333;
            font-size: 18px;
        }}
        .question-text {{
            font-size: 16px;
            color: #555;
            font-weight: 500;
        }}
        .answer-box {{
            background: #e8f5e9;
            padding: 20px;
            border-radius: 8px;
            margin-bottom: 20px;
            border-left: 4px solid #4caf50;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }}
        .answer-box h2 {{
            margin: 0 0 10px 0;
            color: #2e7d32;
            font-size: 18px;
        }}
        .answer-text {{
            font-size: 16px;
            color: #333;
            line-height: 1.6;
        }}
        .context-section {{
            background: white;
            padding: 20px;
            border-radius: 8px;
            margin-bottom: 20px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }}
        .context-section h2 {{
            margin: 0 0 15px 0;
            color: #333;
            font-size: 18px;
            border-bottom: 2px solid #667eea;
            padding-bottom: 10px;
        }}
        .context-item {{
            background: #f5f5f5;
            padding: 15px;
            border-radius: 6px;
            margin-bottom: 15px;
            border-left: 3px solid #667eea;
        }}
        .context-item:last-child {{
            margin-bottom: 0;
        }}
        .context-source {{
            font-size: 12px;
            color: #666;
            margin-bottom: 8px;
            font-weight: 600;
        }}
        .context-title {{
            font-size: 14px;
            color: #333;
            margin-bottom: 8px;
            font-weight: 600;
        }}
        .context-content {{
            font-size: 14px;
            color: #555;
            line-height: 1.5;
            max-height: 150px;
            overflow-y: auto;
            white-space: pre-wrap;
        }}
        .badge {{
            display: inline-block;
            padding: 4px 8px;
            background: #667eea;
            color: white;
            border-radius: 4px;
            font-size: 11px;
            margin-right: 8px;
        }}
    </style>
    
    <div class="rag-container">
        <div class="rag-header">
            <h1>🔍 RAG (Retrieval-Augmented Generation) Results</h1>
        </div>
        
        <div class="question-box">
            <h2>❓ Question</h2>
            <div class="question-text">{escape_html(question)}</div>
        </div>
        
        <div class="answer-box">
            <h2>💡 Answer</h2>
            <div class="answer-text">{escape_html(answer)}</div>
        </div>
        
        <div class="context-section">
            <h2>📚 Retrieved Context ({len(processed_docs)} documents)</h2>
    """
    
    for i, doc in enumerate(processed_docs, 1):
        content_preview = doc['content'][:300] + ('...' if len(doc['content']) > 300 else '')
        html_content += f"""
            <div class="context-item">
                <span class="badge">Document {i}</span>
                <div class="context-source">🔗 Source: {escape_html(doc['source'])}</div>
                <div class="context-title">📄 {escape_html(doc['title'])}</div>
                <div class="context-content">{escape_html(content_preview)}</div>
            </div>
        """
    
    html_content += """
        </div>
    </div>
    """
    
    return html_content

# Example: Use with actual RAG result from cell 22
# Uncomment the line below to use the actual result:
# result = rag_chain_with_source.invoke("What areas of AI should students consider?")
# display(HTML(format_rag_results_html(result)))


display(HTML(format_rag_results_html(result)))

## Limitations and responsible use

- **Source quality:** RAG is only as good as its sources. A page that returns no text, an out-of-date article, or a biased source will give weaker answers.
- **Retrieval misses:** if the right chunk is not in the top 4 results, the model never sees it. Try different questions and check which chunks were retrieved.
- **Hallucination is reduced, not removed:** the model can still add claims that the sources do not support.
- **Privacy:** questions and retrieved text are sent to OpenRouter and the model provider (see the top of this notebook). Do not include personal or confidential data.
- **Copyright and terms:** loading a web page copies its text. Check the site's terms of use and the copyright before you reuse the content outside this course.
- **High-stakes use:** medical, legal and financial answers need checking by a qualified expert, however good the retrieval looks.
